# F1 SE blind-baselines — per-SNR explorer

Interactive per-SNR view of the five speech-enhancement blind baselines
(DCUNet, Edge-BS-RoFormer, MP-SENet, TF-GridNet, SGMSE+), both training passes
(**A** = drone-only, **B** = all-harmonic), on the two fixed valid sets.

Metrics follow the drone-SE survey (Wang et al.): **SI-SDR**, **eSTOI**, **PESQ**.

Backing data = the per-clip CSVs from `scripts/eval_se_perclip.py` (one row per
clip). Pick a valid set, any subset of noise **categories**, any subset of
**models**, and the panels redraw as per-SNR means with the *noisy* and *Wiener*
anchors overlaid.

> **Sync first.** The model CSVs come from the cluster job `f1-perclip-eval-*`:
> `omnirun pull f1-perclip-eval-<id>` (or they land in `results/f1_perclip/` via
> R2). Anchors are computed locally. Re-run the load cell after syncing.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 100

from se_baselines_explorer import load_perclip, explorer, plot_metrics, aggregate

df = load_perclip()
print(f"{len(df):,} clip-rows loaded")
df.groupby(["valid", "method"]).size().unstack(fill_value=0)

## Interactive explorer

Multi-select works with **Ctrl/Cmd-click** (categories, models, metrics).

In [ ]:
explorer(df)

## Static example — the diversity question on drone noise

Pass A vs Pass B for the strong models on the drone category (does diverse
harmonic training help *drone* enhancement?).

In [ ]:
fig = plot_metrics(
    df,
    valid="SE-valid-harmonic",
    categories=["drone"],
    methods=["f1_mpsenet_a", "f1_mpsenet_b", "f1_tfgridnet_a", "f1_tfgridnet_b"],
    show_anchors=True,
)
plt.show()

## Per-category floor (Pass B, one model)

Which noise families are recoverable at all — tonal (motors/aircraft/horns) vs
stochastic (MIMII).

In [ ]:
fig = plot_metrics(
    df,
    valid="SE-valid-harmonic",
    categories=["drone", "motors", "aircraft", "horns", "mimii", "mimii_dg"],
    methods=["f1_mpsenet_b"],
    metrics=["estoi"],
    show_anchors=True,
)
plt.show()